# Parte III. Vecinos cercanos

Se utiliza como insumo la base limpia `B_Outputs/B_Data/EPH_T126_limpia.csv`.

Según corresponda, se analizará en cada caso el tratamiento de valores faltantes. Ante la presencia de `NaN` entre los predictores, se informa las variables afectadas y para la decisión de cómo adaptar el modelado.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score

In [ ]:
# Funciona tanto si VS Code abre el notebook desde C_Programs como desde TP_01.
nombre_base = "EPH_T126_limpia.csv"
carpetas = [Path.cwd(), *Path.cwd().parents]
main_path = next(
    (p for p in carpetas if (p / "B_Outputs" / "B_Data" / nombre_base).is_file()),
    None
)
if main_path is None:
    raise FileNotFoundError(
        f"No se encuentra {nombre_base}. Abrí en VS Code la carpeta del repositorio "
        "y ejecutá este notebook desde TP_01 o C_Programs."
    )

archivo_base = main_path / "B_Outputs" / "B_Data" / nombre_base
carpeta_tablas = main_path / "B_Outputs" / "B_Table"
carpeta_figuras = main_path / "B_Outputs" / "B_Figure"
carpeta_tablas.mkdir(parents=True, exist_ok=True)
carpeta_figuras.mkdir(parents=True, exist_ok=True)

base = pd.read_csv(archivo_base)
print("Base:", archivo_base)
print("Observaciones:", len(base), "| Columnas:", base.shape[1])

In [ ]:
# Se conserva la partición train/test y la variable dependiente de la Parte I.
columnas_requeridas = {"informal", "conjunto"}
assert columnas_requeridas.issubset(base.columns), (
    "La base debe tener las columnas 'informal' y 'conjunto'."
)
assert set(base["conjunto"].dropna().unique()) == {"train", "test"}, (
    "La columna 'conjunto' debe contener únicamente train y test, con ambas presentes."
)
assert not base[["informal", "conjunto"]].isna().any().any(), (
    "Hay faltantes en la variable dependiente o en la partición."
)
assert set(base["informal"].unique()).issubset({0, 1}), (
    "La variable informal debe ser binaria: 0 formal, 1 informal."
)

# Se excluyen la variable dependiente, la partición y el intercepto de logit.
columnas_no_x = ["informal", "conjunto", "const"]
predictores = [col for col in base.columns if col not in columnas_no_x]
assert predictores, "No se encontraron variables predictoras."
assert all(pd.api.types.is_numeric_dtype(base[col]) for col in predictores), (
    "Todos los predictores deben ser numéricos (dummies para las categóricas)."
)

train = base.loc[base["conjunto"] == "train"]
test = base.loc[base["conjunto"] == "test"]
X_train = train[predictores].copy()
y_train = train["informal"].astype(int).copy()
X_test = test[predictores].copy()
y_test = test["informal"].astype(int).copy()

# KNN no acepta NaN: por ahora no se decide cómo imputar o excluir.
faltantes = X_train.isna().sum()

if (faltantes > 0).any():
    print("Variables con faltantes en entrenamiento:")
    print(faltantes.loc[faltantes > 0].sort_values(ascending=False))
    raise ValueError(
        "Hay NaN en X_train. Es necesario definir su tratamiento antes de estimar KNN."
    )

assert np.isfinite(X_train.to_numpy(dtype=float)).all(), (
    "Hay valores infinitos en los predictores de entrenamiento."
)
assert len(X_train) > 100, "Se necesitan más de 100 observaciones de entrenamiento."

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Proporción informal en train:", round(y_train.mean(), 4))
print("Predictores:", len(predictores))

## Inciso 8. KNN con K = 10, 50 y 100

Siguiendo la tutorial 3, y dado que las distancias importan, se estandarizan las variables antes de estimar. `StandardScaler` calcula la media y el desvío de cada variable usando únicamente el entrenamiento. Las variables indicadoras (dummies) también se estandarizan, igual que los demás predictores.

El *accuracy* de entrenamiento que se muestra sirve como descripción del ajuste dentro de la muestra.

In [ ]:
# Medias y desvíos: se actualizan automaticamente cada vez que se corre el notebook.
escala = StandardScaler().fit(X_train)
Z_train = escala.transform(X_train)

# Tabla de referencia: desviación poblacional (ddof=0), como StandardScaler.
resumen_escala = pd.DataFrame({
    "media_train": escala.mean_,
    "desvio_train": np.sqrt(escala.var_)
}, index=predictores)
print("Medias y desvíos calculados con train (primeras variables):")
display(resumen_escala.head(10).round(4))

In [ ]:
# K a considerar dada la consigna.
valores_k = [10, 50, 100]
modelos_knn = {}
resultados_8 = []

for k in valores_k:
    modelo = KNeighborsClassifier(n_neighbors=k).fit(Z_train, y_train)
    modelos_knn[k] = modelo
    acc_train = accuracy_score(y_train, modelo.predict(Z_train))
    resultados_8.append({"K": k, "accuracy_train": acc_train})

tabla_8 = pd.DataFrame(resultados_8)
display(tabla_8.round(4))
tabla_8.to_csv(carpeta_tablas / "tabla_knn_k_fijos.csv", index=False)
print("Tabla del inciso 8 guardada en B_Outputs/B_Table.")

## Inciso 9. Selección de K mediante cross validation
Para obtener el K optimo por *cross validation*, se considera un fold igual a *5*.

Se considera una grilla con valores de K grandes para comprobar que el accuracy deja de mejorar. Si la base definitiva cambia sustancialmente y el máximo vuelve a quedar en el extremo derecho, habría que ampliar la grilla.

In [ ]:
particiones = KFold(n_splits=5, shuffle=True, random_state=444)

# Grilla basada en el Tutorial 4, extendida para observar la zona donde
# el accuracy deja de aumentar. Se adapta si la muestra definitiva es menor.
k_candidatos = [1, 3, 5, 10, 15, 20, 25, 30, 40, 50,
                75, 100, 150, 200, 300, 500, 750, 1000]
menor_fold = len(X_train) - int(np.ceil(len(X_train) / 5))
grilla_k = [k for k in k_candidatos if k <= menor_fold]
assert grilla_k, "No hay valores de K válidos para la cantidad de observaciones."

filas = []
for k in grilla_k:
    for bloque, (estima, valida) in enumerate(particiones.split(X_train), start=1):
        X_e = X_train.iloc[estima]
        X_v = X_train.iloc[valida]
        y_e = y_train.iloc[estima]
        y_v = y_train.iloc[valida]

        # Medias y desvíos calculados exclusivamente dentro de cada fold.
        escala_bloque = StandardScaler().fit(X_e)
        Z_e = escala_bloque.transform(X_e)
        Z_v = escala_bloque.transform(X_v)

        knn = KNeighborsClassifier(n_neighbors=k).fit(Z_e, y_e)
        acc = accuracy_score(y_v, knn.predict(Z_v))
        filas.append({"K": k, "bloque": bloque, "accuracy": acc})

cv = pd.DataFrame(filas)
display(cv.head(10).round(4))

In [ ]:
# Promedio de accuracy entre los cinco bloques para cada valor de K.
accuracy_k = cv.groupby("K")["accuracy"].agg(["mean", "std"])
accuracy_k.columns = ["accuracy_promedio", "desvio_entre_folds"]
k_cv = int(accuracy_k["accuracy_promedio"].idxmax())

print("K óptimo según CV:", k_cv)
print("Accuracy promedio de CV:", round(accuracy_k.loc[k_cv, "accuracy_promedio"], 4))
display(accuracy_k.round(4))

accuracy_k.to_csv(carpeta_tablas / "tabla_knn_cv.csv")
cv.to_csv(carpeta_tablas / "tabla_knn_cv_folds.csv", index=False)

In [ ]:
# Gráfico de accuracy
accuracy_mayoria = max(y_train.mean(), 1 - y_train.mean())

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(accuracy_k.index, accuracy_k["accuracy_promedio"], marker="o")
ax.axvline(k_cv, linestyle="--", color="gray", label=f"K elegido = {k_cv}")
ax.axhline(accuracy_mayoria, linestyle=":", color="gray",
           label="Predicción de clase mayoritaria")
ax.set_xscale("log")
ax.set_xlabel("K (número de vecinos, escala logarítmica)")
ax.set_ylabel("Accuracy promedio (5 folds)")
ax.set_title("KNN: selección de K mediante validación cruzada")
ax.legend()
fig.tight_layout()
fig.savefig(carpeta_figuras / "knn_accuracy_cv.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Modelo KNN final seleccionado por CV: se estima con todo el entrenamiento.
knn_cv = KNeighborsClassifier(n_neighbors=k_cv).fit(Z_train, y_train)
print(f"KNN con K-CV estimado con {len(X_train)} observaciones de entrenamiento.")

# Aviso metodológico si la grilla termina mientras el accuracy sigue mejorando.
if k_cv == max(grilla_k):
    print("ATENCIÓN: el K elegido está en el extremo derecho de la grilla; "
          "conviene ampliar los K evaluados.")
else:
    print("El K elegido no está en el extremo derecho de la grilla.")

### Archivos generados

- `B_Outputs/B_Table/tabla_knn_k_fijos.csv`: resultados descriptivos del inciso 8.
- `B_Outputs/B_Table/tabla_knn_cv_folds.csv`: accuracy de cada K y fold.
- `B_Outputs/B_Table/tabla_knn_cv.csv`: accuracy promedio y desvío entre folds para cada K.
- `B_Outputs/B_Figure/knn_accuracy_cv.png`: gráfico del inciso 9.